# Riesgo crediticio de ONs vs balances CNV

Cruza el universo de Obligaciones Negociables (con TIR ya calculada) con los balances trimestrales reportados por las empresas a la CNV, para detectar ONs de emisores con leverage alto (Deuda Financiera / EBITDA).

**Pipeline:**
1. Carga `output/on_tir.parquet` (generado por `on_tir_dotplot.ipynb`) y `balance_data_html.pkl`.
2. Pivotea cuentas clave del plan CNV por empresa, eligiendo el balance más reciente (preferencia: CONSOLIDADO sobre INDIVIDUAL).
3. Matchea descripciones de ONs (PPI) con nombres de empresas (CNV) usando normalización + tokens + alias manuales.
4. Muestra tabla de ONs ordenadas por riesgo + scatter TIR vs Deuda/EBITDA.

**Caveat:** La cuenta CNV `8000015 Deuda Financiera / EBITDA` usa el EBITDA **del período**, no LTM. Para balances trimestrales (periodicity=3) eso da ratios artificialmente altos. Filtramos a balances anuales (periodicity=1) cuando están disponibles, con fallback al ratio crudo del último trimestre.

In [58]:
import re
import json
import unicodedata
from pathlib import Path

import pandas as pd
import plotly.express as px

## 1. Cargar datos

In [59]:
ROOT = Path("..")
tir_path = ROOT / "output" / "on_tir.parquet"
bal_path = ROOT / "balance_data_html.pkl"

if not tir_path.exists():
    raise FileNotFoundError(
        f"No existe {tir_path}. Corré primero on_tir_dotplot.ipynb hasta el final "
        "(la celda 6 guarda el parquet)."
    )

ons = pd.read_parquet(tir_path)
balances = pd.read_pickle(bal_path)

print(f"ONs con TIR:        {len(ons)}")
print(f"Filas de balance:   {len(balances):,}")
print(f"Empresas en CNV:    {balances['company'].nunique()}")
print(f"Último cierre CNV:  {balances['close_date'].max().date()}")

ONs con TIR:        110
Filas de balance:   287,849
Empresas en CNV:    213
Último cierre CNV:  2026-09-30


## 2. Pivotar cuentas clave por empresa

Para cada empresa, tomamos el **balance más reciente** (preferentemente CONSOLIDADO; si no, INDIVIDUAL) y extraemos:
- Indicadores CNV pre-calculados: `8000015 Deuda Financiera / EBITDA`, `8000016 EBITDA / Intereses`, `8000006 Liquidez`.
- Cuentas crudas para contexto: EBITDA, ON emitidas, Pasivos Financieros (CP+LP), Efectivo, Patrimonio Neto, Activo Total.

También priorizamos balances ANUALES (periodicity=1) para el ratio Deuda/EBITDA, ya que el EBITDA trimestral subestima la generación de caja.

In [60]:
KEY_ACCOUNTS = {
    "8000015": "deuda_fin_sobre_ebitda",
    "8000016": "ebitda_sobre_intereses",
    "8000006": "liquidez",
    "8000004": "ebitda",
    "2311600": "on_emitidas",
    "2322200": "pas_fin_corriente",
    "2312300": "pas_fin_no_corriente",
    "1122500": "efectivo",
    "2299999": "patrimonio_neto",
    "1999999": "total_activo",
    "2399999": "total_pasivo",
}


def parse_amount(s):
    if s is None or pd.isna(s) or str(s).strip() in {"-", ""}:
        return pd.NA
    try:
        return float(str(s).replace(",", ""))
    except ValueError:
        return pd.NA


bal = balances.copy()
bal["account_number"] = bal["account_number"].astype(str).str.strip().str.split(".").str[0]
bal = bal[bal["account_number"].isin(KEY_ACCOUNTS) & bal["company"].notna()].copy()
bal["value"] = bal["ammount"].map(parse_amount)
bal = bal.dropna(subset=["value"])

# Para cada (company, account, periodicity), nos quedamos con el cierre más reciente,
# y entre INDIVIDUAL/CONSOLIDADO preferimos CONSOLIDADO (incluye subsidiarias y por ende deuda real).
bal["_consol_rank"] = (bal["tipo_balance"] == "CONSOLIDADO").astype(int)
bal = bal.sort_values(["company", "account_number", "close_date", "_consol_rank"], ascending=[True, True, False, False])

# Para cada cuenta tenemos dos perspectivas: ANUAL (periodicity=1) y la última disponible.
anual = bal[bal["periodicity"].astype(str).isin(["1", "ANUAL"])].drop_duplicates(["company", "account_number"])
ultimo = bal.drop_duplicates(["company", "account_number"])

anual_piv = anual.pivot_table(index="company", columns="account_number", values="value", aggfunc="first")
ultimo_piv = ultimo.pivot_table(index="company", columns="account_number", values="value", aggfunc="first")

anual_piv = anual_piv.rename(columns=KEY_ACCOUNTS)
ultimo_piv = ultimo_piv.rename(columns=KEY_ACCOUNTS)

# Marca de fecha del balance usado como referencia
ref_date_anual = anual.drop_duplicates("company").set_index("company")["close_date"].rename("fecha_balance_anual")
ref_date_ultimo = ultimo.drop_duplicates("company").set_index("company")["close_date"].rename("fecha_balance_ultimo")
ref_period_ultimo = ultimo.drop_duplicates("company").set_index("company")["periodicity"].rename("period_ultimo")

# DataFrame consolidado por empresa: usamos ratio anual si existe, sino el último
company_metrics = pd.DataFrame(index=ultimo_piv.index)
company_metrics["deuda_fin_sobre_ebitda"] = anual_piv["deuda_fin_sobre_ebitda"].combine_first(ultimo_piv.get("deuda_fin_sobre_ebitda"))
company_metrics["deuda_ebitda_es_anual"] = anual_piv["deuda_fin_sobre_ebitda"].notna()
for col in ["ebitda_sobre_intereses", "liquidez", "ebitda", "on_emitidas", "pas_fin_corriente",
            "pas_fin_no_corriente", "efectivo", "patrimonio_neto", "total_activo", "total_pasivo"]:
    if col in ultimo_piv.columns:
        company_metrics[col] = ultimo_piv[col]
company_metrics["deuda_fin_total"] = company_metrics[["pas_fin_corriente", "pas_fin_no_corriente"]].sum(axis=1, min_count=1)
company_metrics = company_metrics.join(ref_date_anual).join(ref_date_ultimo).join(ref_period_ultimo)

print(f"Empresas con métricas:    {len(company_metrics)}")
print(f"Con Deuda/EBITDA:         {company_metrics['deuda_fin_sobre_ebitda'].notna().sum()}")
print(f"De las cuales son anuales: {company_metrics['deuda_ebitda_es_anual'].sum()}")
company_metrics.head(10)

Empresas con métricas:    211
Con Deuda/EBITDA:         154
De las cuales son anuales: 150


,deuda_fin_sobre_ebitda,deuda_ebitda_es_anual,ebitda_sobre_intereses,liquidez,ebitda,on_emitidas,pas_fin_corriente,pas_fin_no_corriente,efectivo,patrimonio_neto,total_activo,total_pasivo,deuda_fin_total,fecha_balance_anual,fecha_balance_ultimo,period_ultimo
company,,,,,,,,,,,,,,,,
360 ENERGY SOLAR S.A.,6.85,True,-1.49,1.48,25075645.0,NaN,19094542.0,152693181.0,414022.0,91983234.0,300421809.0,208438575.0,171787723.0,2025-12-31,2025-12-31,1
A3 Mercados S.A.,3.25,True,-12.69,1.84,28021.0,NaN,117613.0,1199.0,223573.0,637674.0,873874.0,236200.0,118812.0,2025-06-30,2025-09-30,3
AES ARGENTINA GENERACIÓN S.A.,1.62,True,-12.93,0.75,51123.0,NaN,113194.0,42541.0,51859.0,574764.0,943927.0,369163.0,155735.0,2025-12-31,2026-03-31,3
ARCOR SAIC,3.52,True,-1.98,1.46,408705.27,NaN,669344.0,769081.0,294507.0,1641652.0,4402878.0,2761226.0,1438425.0,2025-12-31,2025-12-31,1
Aeropuertos Argentina 2000 S.A.,1.72,True,-3.81,0.89,511263454201.0,NaN,219009426815.0,659125183948.0,94184002910.0,1535617949491.0,3047492016608.0,1511874067117.0,878134610763.0,2025-12-31,2025-12-31,1
Agrofina S.A.,-565.99,False,-0.28,2.3,-219907.87,NaN,1102688.0,123362749.0,6960263.0,-41380660.0,191050311.0,232430971.0,124465437.0,2025-06-30,2025-12-31,3
Agrometal Sociedad Anonima Industrial,2.09,True,1.01,1.69,-832386657.0,NaN,27314573987.0,4558194073.0,9908858729.0,47181314235.0,85428976301.0,38247662066.0,31872768060.0,2025-12-31,2026-03-31,3
Aluar Aluminio Argentino,3.74,True,-2.12,2.24,191998760335.0,31900079.0,541574749726.0,451401745362.0,101435234162.0,1845601131948.0,3247183794585.0,1401582662637.0,992976495088.0,2025-06-30,2025-12-31,3
Angel Estrada y Cia.,0.43,True,-1.22,1.98,2724768390.0,NaN,10542068845.0,1950491004.0,1112186967.0,51429681114.0,93797137512.0,42367456398.0,12492559849.0,2025-06-30,2025-12-31,3


## 3. Matchear ONs con empresas

Estrategia:
1. **Normalización**: uppercase, sin acentos, sin sufijos societarios (`S.A.`, `SAU`, `SACIF`, etc.), sin puntuación.
2. **Alias manuales** para acrónimos de mercado que no son obvios (`TGS` → `Transportadora de Gas del Sur S.A.`, `IRSA` → `IRSA INVERSIONES Y REPRESENTACIONES SA`, etc.).
3. **Match**: la descripción de la ON tiene que *empezar* con el nombre normalizado de la empresa, o tener overlap alto de los primeros tokens.
4. Las que no matchean quedan en una tabla aparte para revisar manualmente.

In [61]:
# Tokens corporativos / de relleno que no aportan al match
SUFFIX_TOKENS = {
    "ON","SA","SAS","SAU","SACIFI","SACIF","SACIFIA","SAIC","SAICF","SAICFA","SAICFIA",
    "SAICA","SRL","SCA","SH","CIASA","CIA","CIASAU",
    "AGICI","AGICIYF","AGI","AGIC",
    "SOCIEDAD","ANONIMA","COMERCIAL","INDUSTRIAL","FINANCIERA",
    "DE","DEL","LA","LAS","LOS","EL","Y",
}


def strip_accents(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")


def normalize_name(s: str) -> str:
    """Normaliza nombres de empresas / descripciones de ONs a tokens comparables.

    - Mayúsculas + sin acentos.
    - Colapsa abreviaturas con puntos: 'S.A.' → 'SA', 'A.G.I.C.I.' → 'AGICI'.
    - Limpia puntuación.
    - Quita sufijos societarios y stop-words (SA, SAU, DE, Y, etc.).
    - Quita tokens de 1 letra (ruido tipo 'E', 'A', 'F').
    """
    if not isinstance(s, str):
        return ""
    s = strip_accents(s).upper()
    # Colapsa runs de letra-punto contiguos: 'S.A.' → 'SA', sin tragar el espacio que sigue.
    s = re.sub(r"\b(?:[A-Z]\.){2,}", lambda m: m.group(0).replace(".", ""), s)
    s = re.sub(r"[\.,;:'\"`´\(\)\[\]/\\]", " ", s)
    s = re.sub(r"[^A-Z0-9& ]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    parts = [p for p in s.split() if p not in SUFFIX_TOKENS and len(p) > 1]
    return " ".join(parts)


# Aliases de mercado: clave = prefijo en la descripción de la ON, valor = nombre normalizado
# en CNV (después de normalize_name). Las claves son matched longest-first así
# 'IRSA PROPIEDADES' gana sobre 'IRSA'. Editá esto cuando aparezcan ONs sin match.
MARKET_ALIASES = {
    # Energía / Oil & Gas
    "TGS": "TRANSPORTADORA GAS SUR",
    "TGN": "TRANSPORTADORA GAS NORTE",
    "TRANSPORTADORA GAS DEL SUR": "TRANSPORTADORA GAS SUR",
    "TRANSPORTADORA GAS DEL NORTE": "TRANSPORTADORA GAS NORTE",
    "PAMPA ENERGIA": "PAMPA ENERGIA",
    "PAMPA": "PAMPA ENERGIA",
    "YPF LUZ": "YPF ENERGIA ELECTRICA",
    "YPF ENERGIA ELECTRICA": "YPF ENERGIA ELECTRICA",
    "YPF ENERGIA": "YPF ENERGIA ELECTRICA",
    "YPF": "YPF",
    "VISTA ENERGY ARGENTINA": "VISTA ENERGY ARGENTINA",
    "VISTA OIL & GAS": "VISTA ENERGY ARGENTINA",
    "VISTA OIL": "VISTA ENERGY ARGENTINA",
    "VISTA": "VISTA ENERGY ARGENTINA",
    "AES ARGENTINA GENERACION": "AES ARGENTINA GENERACION",
    "AES ARGENTINA": "AES ARGENTINA GENERACION",
    "AES": "AES ARGENTINA GENERACION",
    "PETROLEOS SUDAMERICANOS": "PETROLEOS SUDAMERICANOS",
    "PETROLERA ACONCAGUA": "PETROLERA ACONCAGUA ENERGIA",
    "PAN AMERICAN ENERGY": "PAN AMERICAN ENERGY",
    "CGC": "GENERAL COMBUSTIBLES",
    "COMPANIA GENERAL DE COMBUSTIBLES": "GENERAL COMBUSTIBLES",
    "GENERAL DE COMBUSTIBLES": "GENERAL COMBUSTIBLES",
    "GENNEIA": "GENNEIA",
    "GENERACION MEDITERRANEA": "GENERACION MEDITERRANEA",
    "MSU ENERGY": "MSU ENERGY",
    "MSU GREEN ENERGY": "MSU GREEN ENERGY",
    "MSU": "MSU",
    "360 ENERGY": "360 ENERGY SOLAR",
    "CAPEX": "CAPEX",
    "CENTRAL PUERTO": "CENTRAL PUERTO",
    "CENTRAL COSTANERA": "CENTRAL COSTANERA",
    "EDENOR": "EMPRESA DISTRIBUIDORA COMERCIALIZADORA NORTE",
    "EMPRESA DISTRIBUIDORA NORTE": "EMPRESA DISTRIBUIDORA COMERCIALIZADORA NORTE",
    "METROGAS": "METROGAS",
    "ALBANESI ENERGIA": "GENERACION MEDITERRANEA",  # Albanesi opera vía Generación Mediterránea
    "ALBANESI": "GENERACION MEDITERRANEA",
    # Industria / Consumo
    "ALUAR ALUMINIO ARGENTINO": "ALUAR ALUMINIO ARGENTINO",
    "ALUAR": "ALUAR ALUMINIO ARGENTINO",
    "ARCOR": "ARCOR",
    "MASTELLONE": "MASTELLONE HERMANOS",
    "LOMA NEGRA": "LOMA NEGRA",
    "MOLINOS RIO DE LA PLATA": "MOLINOS RIO PLATA",
    "MOLINOS AGRO": "MOLINOS AGRO",
    "MOLINOS": "MOLINOS AGRO",
    "FERRUM": "FERRUM CERAMICA METALURGIA",
    "RICHMOND": "LABORATORIOS RICHMOND",
    "BOLDT": "BOLDT",
    # Agro / Real estate
    "CRESUD": "CRESUD",
    "IRSA PROPIEDADES COMERCIALES": "IRSA PROPIEDADES COMERCIALES",
    "IRSA PROPIEDADES": "IRSA PROPIEDADES COMERCIALES",
    "IRSA INVERSIONES": "IRSA INVERSIONES REPRESENTACIONES",
    "IRSA": "IRSA INVERSIONES REPRESENTACIONES",
    "RAGHSA": "RAGHSA",
    "SAN MIGUEL": "SAN MIGUEL",
    # Telecom / media
    "TELECOM ARGENTINA": "TELECOM ARGENTINA",
    "TELECOM ARG": "TELECOM ARGENTINA",
    "TELECOM": "TELECOM ARGENTINA",
    "CABLEVISION": "CABLEVISION HOLDING",
    # Bancos
    "BANCO MACRO": "BANCO MACRO",
    "BANCO HIPOTECARIO": "BANCO HIPOTECARIO",
    "BANCO SUPERVIELLE": "BANCO SUPERVIELLE",
    "BANCO SANTANDER ARGENTINA": "BANCO SANTANDER ARGENTINA",
    "BANCO SANTANDER": "BANCO SANTANDER ARGENTINA",
    "BANCO COMAFI": "BANCO COMAFI",
    "BANCO BBVA ARGENTINA": "BANCO BBVA ARGENTINA",
    "BANCO BBVA": "BANCO BBVA ARGENTINA",
    "BBVA BANCO FRANCES": "BANCO BBVA ARGENTINA",
    "BBVA": "BANCO BBVA ARGENTINA",
    "BANCO PIANO": "BANCO PIANO",
    "BANCO MARIVA": "BANCO MARIVA",
    "BANCO GALICIA Y BUENOS AIRES": "BANCO GALICIA BUENOS AIRES",
    "BANCO GALICIA": "BANCO GALICIA BUENOS AIRES",
    "GRUPO FINANCIERO GALICIA": "GRUPO FINANCIERO GALICIA",
    "GALICIA": "GRUPO FINANCIERO GALICIA",
    # Distribuidoras / industria con abreviaturas de mercado
    "EDEMSA":              "EMP DISTRIBUIDORA ELECTRICIDAD MENDOZA",
    "PETROQUIMICA COMODORO": "PETROQUIMICA COMODORO RIVADAVIA",
    "PETROQUIMICA":        "PETROQUIMICA COMODORO RIVADAVIA",
    "GMCTR":               "GENERACION MEDITERRANEA",
    "AA2000":              "AEROPUERTOS ARGENTINA 2000",
    "AEROP ARG 2000":      "AEROPUERTOS ARGENTINA 2000",
    "AEROPUERTOS ARGENTINA 2000": "AEROPUERTOS ARGENTINA 2000",
    "SIDERSA":             "SIDERSA",
    "OILTANKING EBYTEM":   "OILTANKING EBYTEM",
    "RIZOBACTER":          "RIZOBACTER ARGENTINA",
    "INVERSIONES JURAMENTO": "INVERSORA JURAMENTO",
    "INV JURAMENTO":       "INVERSORA JURAMENTO",
    "PLAZA LOGISTICA":     "PLAZA LOGISTICA",
    "LIPSA":               "LIPSA",
    # Camuzzi: ZZC ticker — Pampeana es el emisor más grande del grupo
    "CAMUZZI":             "CAMUZZI GAS PAMPEANA",
    # EDESA = Edesa Holding (distribuidora Salta/Jujuy)
    "EDESA":               "EDESA HOLDING",
    # PCR = Petroquímica Comodoro Rivadavia (ticker alternativo)
    "PCR":                 "PETROQUIMICA COMODORO RIVADAVIA",
    # GENERA. LITORAL = abreviatura de Generación Litoral
    "GENERA LITORAL":      "GENERACION LITORAL",
    "GENERACION LITORAL":  "GENERACION LITORAL",
    # 360 ENER.SOL. = 360 Energy Solar (abreviatura en PPI)
    "360 ENER SOL":        "360 ENERGY SOLAR",
    "360 ENER":            "360 ENERGY SOLAR",
}


def find_company(desc: str, company_keys: dict, aliases: dict):
    """Devuelve (nombre_original_CNV, método_de_match) o (None, 'no_match')."""
    n = normalize_name(desc)
    if not n:
        return None, "empty"

    # 1) Aliases longest-first → buscamos exact match del target en CNV, después prefijo, después sub
    for alias, target in sorted(aliases.items(), key=lambda x: -len(x[0])):
        if n == alias or n.startswith(alias + " "):
            exact = [orig for orig, key in company_keys.items() if key == target]
            if exact:
                return exact[0], f"alias_exact:{alias}"
            prefix = [orig for orig, key in company_keys.items() if key and key.startswith(target + " ")]
            if prefix:
                return min(prefix, key=lambda o: len(company_keys[o])), f"alias_prefix:{alias}"
            sub = [orig for orig, key in company_keys.items() if key and target.startswith(key + " ")]
            if sub:
                return max(sub, key=lambda o: len(company_keys[o])), f"alias_sub:{alias}"

    # 2) Match directo: la descripción empieza con el nombre normalizado de una empresa
    candidates = [(orig, key) for orig, key in company_keys.items() if key and (n.startswith(key + " ") or n == key)]
    if candidates:
        best = max(candidates, key=lambda x: len(x[1]))
        return best[0], f"prefix:{best[1]}"

    # 3) Token-overlap fallback (mínimo 2 tokens en los primeros 3)
    desc_tokens = set(n.split()[:3])
    best, best_score = None, 0
    for orig, key in company_keys.items():
        if not key:
            continue
        comp_tokens = set(key.split()[:3])
        score = len(desc_tokens & comp_tokens)
        if score > best_score:
            best, best_score = orig, score
    if best_score >= 2:
        return best, f"tokens:{best_score}"
    return None, "no_match"


company_keys = {c: normalize_name(c) for c in company_metrics.index}
match_result = ons["descripcion"].apply(
    lambda d: pd.Series(find_company(d, company_keys, MARKET_ALIASES), index=["company", "match_method"])
)
ons_matched = pd.concat([ons, match_result], axis=1)

n_matched = ons_matched["company"].notna().sum()
print(f"ONs matcheadas: {n_matched} / {len(ons_matched)} ({100*n_matched/len(ons_matched):.1f}%)")
print("\nDistribución por método de match:")
print(ons_matched["match_method"].apply(lambda s: s.split(":")[0]).value_counts().to_string())

ONs matcheadas: 110 / 110 (100.0%)

Distribución por método de match:
match_method
alias_exact    93
prefix          9
tokens          8


### Diagnóstico: ONs sin match

Si hay muchas ONs sin match, agregá entradas al diccionario `MARKET_ALIASES` arriba y volvé a correr.

In [62]:
unmatched = ons_matched[ons_matched["company"].isna()][["ticker", "descripcion", "moneda_emision"]].drop_duplicates("descripcion")
print(f"{len(unmatched)} descripciones únicas sin match. Top emisores aparentes (primeros 2 tokens):")
if len(unmatched):
    unmatched["emisor_aprox"] = unmatched["descripcion"].apply(lambda s: " ".join(normalize_name(s).split()[:2]))
    print(unmatched["emisor_aprox"].value_counts().head(20).to_string())
    print("\nEjemplos:")
    print(unmatched.head(20).to_string(index=False))

0 descripciones únicas sin match. Top emisores aparentes (primeros 2 tokens):


## 4. Cruzar TIRs con métricas de riesgo

In [63]:
risk = ons_matched.merge(company_metrics, left_on="company", right_index=True, how="left")

# Bucket de riesgo por Deuda/EBITDA (heurístico, ajustar según industria)
def risk_bucket(x):
    if pd.isna(x):
        return "sin dato"
    if x < 0:
        return "EBITDA negativo"
    if x <= 2:
        return "bajo (≤2x)"
    if x <= 4:
        return "moderado (2-4x)"
    if x <= 6:
        return "alto (4-6x)"
    return "muy alto (>6x)"

risk["riesgo_bucket"] = risk["deuda_fin_sobre_ebitda"].apply(risk_bucket)
print("Distribución de ONs por bucket de riesgo:")
print(risk["riesgo_bucket"].value_counts().to_string())

Distribución de ONs por bucket de riesgo:
riesgo_bucket
moderado (2-4x)    59
muy alto (>6x)     19
sin dato           13
bajo (≤2x)         11
alto (4-6x)         5
EBITDA negativo     3


## 5. Tabla de ONs por riesgo

Ranking por Deuda Financiera / EBITDA descendente. Incluye TIR, vencimiento y métricas auxiliares.
Editá `MIN_TIR` y `MAX_VENCIMIENTO` si querés filtrar.

In [64]:
MIN_TIR = None         # ej: 0.10 para mostrar solo ONs con TIR > 10%
MAX_VENCIMIENTO = None # ej: "2030-12-31"

view = risk.copy()
if MIN_TIR is not None:
    view = view[view["tir"] >= MIN_TIR]
if MAX_VENCIMIENTO is not None:
    view = view[view["fechaVencimiento"] <= pd.Timestamp(MAX_VENCIMIENTO)]

table = (
    view[view["deuda_fin_sobre_ebitda"].notna()]
    .sort_values("deuda_fin_sobre_ebitda", ascending=False)
    [[
        "ticker", "descripcion", "company", "riesgo_bucket",
        "deuda_fin_sobre_ebitda", "deuda_ebitda_es_anual",
        "ebitda_sobre_intereses", "liquidez",
        "tir_pct", "fechaVencimiento", "moneda_cotizacion", "moneda_emision",
        "price", "volumen", "fecha_balance_ultimo",
    ]]
    .reset_index(drop=True)
)

with pd.option_context("display.max_rows", 50, "display.max_colwidth", 60):
    display(table.head(50))

print(f"\nTotal ONs en tabla: {len(table)}")
print(f"En bucket alto/muy alto: {(table['riesgo_bucket'].isin(['alto (4-6x)','muy alto (>6x)','EBITDA negativo'])).sum()}")

,ticker,descripcion,company,riesgo_bucket,deuda_fin_sobre_ebitda,deuda_ebitda_es_anual,ebitda_sobre_intereses,liquidez,tir_pct,fechaVencimiento,moneda_cotizacion,moneda_emision,price,volumen,fecha_balance_ultimo
0,XMC1C,ON MINERA EXAR CL.1 V.11/11/27 U$S CG,MINERA EXAR SA,muy alto (>6x),25.35,True,-0.42,1.42,12.770757,2027-11-11 13:47:10.080,USD CCL,USD,99.000000,1.079100e+02,2025-12-31
1,XMC1D,ON MINERA EXAR CL.1 V.11/11/27 U$S CG,MINERA EXAR SA,muy alto (>6x),25.35,True,-0.42,1.42,9.097851,2027-11-11 13:47:10.080,USD MEP,USD,102.900002,7.099141e+04,2025-12-31
2,SNSDD,ON SAN MIGUEL S.12 V14/07/29 U$S CG,S.A. San Miguel A.G.I.C.I. y F.,muy alto (>6x),22.12,True,-0.51,0.68,28.942842,2029-07-14 13:52:46.047,USD MEP,USD,66.500000,0.000000e+00,2025-12-31
3,RZABD,ON RIZOBACTER S.10 CL.B V28/11/27 U$S CG,Rizobacter Argentina S.A.,muy alto (>6x),22.04,True,-1.92,1.13,37.682540,2027-11-28 19:04:14.583,USD MEP,USD,71.800003,1.503072e+04,2025-12-31
4,GOC4D,ON GENERA. LITORAL CL4 V.28/04/29 U$S CG,GENERACION LITORAL S.A.,muy alto (>6x),20.42,True,-1.0,1.33,11.651807,2029-04-28 20:02:53.007,USD MEP,USD,87.900002,1.758000e+01,2025-12-31
5,SIC1C,ON SIDERSA CL1 V09/12/26,SIDERSA SA,muy alto (>6x),17.76,True,2.68,1.82,16.826918,2026-12-09 19:25:53.160,USD CCL,USD,97.500000,0.000000e+00,2026-02-28
6,SIC1D,ON SIDERSA CL1 V09/12/26,SIDERSA SA,muy alto (>6x),17.76,True,2.68,1.82,6.787933,2026-12-09 19:25:53.160,USD MEP,USD,102.599998,3.898928e+04,2026-02-28
7,JNC5D,ON INV. JURAMENTO CL.5 V09/10/26 U$S CG,Inversora Juramento S.A.,muy alto (>6x),11.49,True,-0.71,1.05,1.444524,2026-10-09 20:16:49.013,USD MEP,USD,102.400002,0.000000e+00,2025-12-31
8,LDCGD,ON LEDESMA CL.15 VTO.04/10/27 U$S CG,Ledesma S.A.,muy alto (>6x),8.72,True,-1.87,2.16,8.000967,2027-10-04 13:41:01.510,USD MEP,USD,99.099998,3.578489e+04,2026-02-28
9,LECGC,ON ALBANESI ENERG. CL.15 V30/08/27 U$S CG,Generacion Mediterranea S.A.,muy alto (>6x),8.35,True,-1.35,0.09,12.845522,2027-08-30 13:10:20.850,USD CCL,USD,98.660698,0.000000e+00,2025-12-31



Total ONs en tabla: 97
En bucket alto/muy alto: 27


## 6. Scatter TIR vs Deuda/EBITDA

Cuadrante interesante: **arriba-izquierda** (TIR alta + leverage bajo) son posibles oportunidades; **abajo-derecha** (TIR baja + leverage alto) son ONs potencialmente caras para el riesgo asumido.

In [65]:
import plotly.graph_objects as go

plot_df = risk[risk["deuda_fin_sobre_ebitda"].notna() & risk["tir_pct"].notna()].copy()
plot_df["deuda_fin_sobre_ebitda_capped"] = plot_df["deuda_fin_sobre_ebitda"].clip(-2, 15)

COLOR_PALETTE = {
    "bajo (≤2x)": "#2ca02c",
    "moderado (2-4x)": "#bcbd22",
    "alto (4-6x)": "#ff7f0e",
    "muy alto (>6x)": "#d62728",
    "EBITDA negativo": "#7f0000",
    "sin dato": "#7f7f7f",
}
present_buckets = [b for b in COLOR_PALETTE if b in plot_df["riesgo_bucket"].unique()]

vol = plot_df["volumen"].fillna(0).clip(lower=0)
max_vol = vol.max() if vol.max() > 0 else 1
marker_size = ((vol / max_vol).pow(0.5) * 35).clip(lower=4)

def _fmt_date(s):
    ts = pd.to_datetime(s, errors="coerce")
    return ts.strftime("%Y-%m-%d") if pd.notna(ts) else ""

fig = go.Figure()
for bucket in present_buckets:
    mask = plot_df["riesgo_bucket"] == bucket
    d = plot_df[mask]
    custom = pd.DataFrame({
        "ticker": d["ticker"],
        "company": d["company"],
        "descripcion": d["descripcion"],
        "tir": d["tir_pct"].map("{:.2f}".format),
        "deuda": d["deuda_fin_sobre_ebitda"].map("{:.2f}".format),
        "vcto": d["fechaVencimiento"].apply(_fmt_date),
        "cotiz": d["moneda_cotizacion"],
        "emis": d["moneda_emision"],
        "vol": d["volumen"].map("{:,.0f}".format),
        "balance": d["fecha_balance_ultimo"].apply(_fmt_date),
    }).values
    fig.add_trace(go.Scatter(
        x=d["deuda_fin_sobre_ebitda_capped"],
        y=d["tir_pct"],
        mode="markers+text",
        name=bucket,
        marker=dict(
            color=COLOR_PALETTE[bucket],
            size=marker_size[mask].values,
            line=dict(width=1, color="DarkSlateGrey"),
        ),
        text=d["ticker"],
        textposition="top center",
        customdata=custom,
        hovertemplate=(
            "<b>%{customdata[0]}</b> — %{customdata[1]}<br>"
            "%{customdata[2]}<br>"
            "TIR: %{customdata[3]}%  |  Deuda/EBITDA: %{customdata[4]}x<br>"
            "Vcto: %{customdata[5]}  |  %{customdata[6]} (%{customdata[7]})<br>"
            "Volumen: %{customdata[8]}  |  Balance: %{customdata[9]}<br>"
            "<extra></extra>"
        ),
    ))

fig.update_layout(
    title="ONs: TIR vs Deuda Financiera / EBITDA<br><sup>tamaño = volumen · color = bucket de riesgo · valores cap a [-2, 15]</sup>",
    xaxis_title="Deuda Financiera / EBITDA (x)",
    yaxis_title="TIR (%)",
    yaxis_ticksuffix="%",
    height=650,
    template="plotly_white",
)
fig.add_vline(x=4, line_dash="dash", line_color="orange", annotation_text="4x", annotation_position="top")
fig.add_vline(x=6, line_dash="dash", line_color="red", annotation_text="6x", annotation_position="top")
fig.show()

## 7. ONs candidatas: TIR 7–9 %, USD hard-dollar, vto > ene-2027

Filtramos el universo completo de 110 ONs buscando aquellas que:
- Pagan en **USD (hard-dollar)**
- Tienen **TIR entre 7 % y 9 %**
- Vencen **después del 01-ene-2027**
- El emisor **no** está en bucket de riesgo *"muy alto (>6x)"* ni *"EBITDA negativo"*

Las ONs sin balance CNV disponible quedan marcadas como `sin dato` — incluidas en el análisis pero sin métricas de leverage verificadas.

### Dashboard interactivo

Para cada candidata el dashboard muestra:
- **Panel izquierdo:** empresa emisora — Deuda Fin/EBITDA, EBITDA/Intereses, Liquidez, fecha del balance.
- **Panel derecho:** datos del bono — ticker, precio, TIR, vencimiento, mercado de cotización.
- **Gráfico inferior:** flujo de pagos futuros desagregado en **intereses** (azul) y **amortización** (verde), por cada 100 de VN en USD.

Usá el dropdown para navegar entre ONs.

In [ ]:
# ── parámetros de filtro ──────────────────────────────────────────────────────
MIN_TIR_PCT  = 7.0
MAX_TIR_PCT  = 9.0
MIN_VTO      = pd.Timestamp("2027-01-01")
EXCL_BUCKETS = {"EBITDA negativo", "muy alto (>6x)"}

candidatas = (
    risk[
        (risk["moneda_emision"] == "USD") &
        (risk["tir_pct"].between(MIN_TIR_PCT, MAX_TIR_PCT)) &
        (risk["fechaVencimiento"] >= MIN_VTO) &
        (~risk["riesgo_bucket"].isin(EXCL_BUCKETS))
    ]
    .sort_values("tir_pct", ascending=False)
    .reset_index(drop=True)
)


def _fmt_candidata(r):
    return pd.Series({
        "ticker":        r["ticker"],
        "empresa (CNV)": r["company"] if pd.notna(r["company"]) else "❓ sin match",
        "riesgo":        r["riesgo_bucket"],
        "TIR %":         f"{r['tir_pct']:.2f}%",
        "vencimiento":   str(r["fechaVencimiento"])[:10],
        "precio":        f"{r['price']:.2f}",
        "cotización":    r["moneda_cotizacion"],
        "Deuda/EBITDA":  f"{r['deuda_fin_sobre_ebitda']:.2f}x" if pd.notna(r.get("deuda_fin_sobre_ebitda")) else "N/D",
        "EBITDA/Int":    f"{abs(r['ebitda_sobre_intereses']):.2f}x" if pd.notna(r.get("ebitda_sobre_intereses")) else "N/D",
        "liquidez":      f"{r['liquidez']:.2f}" if pd.notna(r.get("liquidez")) else "N/D",
    })


print(
    f"ONs candidatas — TIR {MIN_TIR_PCT}–{MAX_TIR_PCT}%, "
    f"USD hard-dollar, vto ≥ {MIN_VTO.date()}, sin riesgo muy alto: "
    f"{len(candidatas)}\n"
)
with pd.option_context("display.max_colwidth", 55, "display.max_rows", 20):
    display(candidatas.apply(_fmt_candidata, axis=1))
from pathlib import Path

# Exportar para uso standalone de on_report.py
candidatas.to_parquet(Path("../output/candidatas.parquet"))

In [67]:
import json as _json
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# ── helpers ──────────────────────────────────────────────────────────────────

def _parse_flujos(flujos_raw) -> pd.DataFrame:
    """JSON de flujos → DataFrame con fecha, amortizacion, interes, total (solo pagos futuros)."""
    raw = _json.loads(flujos_raw) if isinstance(flujos_raw, str) else (flujos_raw or [])
    today = pd.Timestamp.now().normalize()
    rows = []
    for f in raw:
        ts = pd.to_datetime(f.get("fechaCorte", ""), errors="coerce")
        if pd.isna(ts):
            continue
        if ts.tzinfo is not None:
            ts = ts.tz_convert("UTC").tz_localize(None)
        fecha = ts.normalize()
        if fecha < today:
            continue
        rows.append({
            "fecha":        fecha,
            "amortizacion": float(f.get("amortizacion") or 0),
            "interes":      float(f.get("interes")      or 0),
            "total":        float(f.get("total")        or 0),
        })
    if not rows:
        return pd.DataFrame(columns=["fecha", "amortizacion", "interes", "total"])
    return pd.DataFrame(rows).sort_values("fecha").reset_index(drop=True)


def _nd(val, suffix="", fmt=".2f") -> str:
    return f"{val:{fmt}}{suffix}" if pd.notna(val) else "N/D"


_RISK_COLORS = {
    "bajo (≤2x)":      "#27ae60",
    "moderado (2-4x)": "#f39c12",
    "alto (4-6x)":     "#e67e22",
    "muy alto (>6x)":  "#e74c3c",
    "EBITDA negativo": "#8e44ad",
    "sin dato":        "#95a5a6",
}
_HDR   = "#1f3a5f"
_CELL  = "#f5f8fa"
_C_INT = "#4472CA"   # azul — intereses
_C_AMT = "#70AD47"   # verde — amortización


# ── dashboard builder ─────────────────────────────────────────────────────────

def build_on_dashboard(cands: pd.DataFrame) -> go.Figure:
    """
    Dashboard interactivo para N ONs candidatas.
    Layout:
      Fila 1 (28 %): tabla empresa  |  tabla bono
      Fila 2 (72 %): stacked bars flujo de pagos (colspan 2)
    Dropdown para cambiar de ON.
    """
    n = len(cands)
    if n == 0:
        raise ValueError("No hay candidatas para mostrar.")

    fig = make_subplots(
        rows=2, cols=2,
        specs=[
            [{"type": "table"}, {"type": "table"}],
            [{"type": "xy", "colspan": 2}, None],
        ],
        subplot_titles=["Empresa", "Bono", "Flujo de pagos (por c/100 VN · USD)"],
        row_heights=[0.28, 0.72],
        vertical_spacing=0.10,
        horizontal_spacing=0.04,
    )

    # ── leyenda permanente (trazas vacías, siempre visibles) ─────────────────
    for name, color, group in [
        ("Intereses",    _C_INT, "interes"),
        ("Amortización", _C_AMT, "amort"),
    ]:
        fig.add_trace(go.Bar(
            name=name, marker_color=color,
            legendgroup=group, showlegend=True,
            x=[], y=[], visible=True,
        ), row=2, col=1)
    LEGEND_OFFSET = 2   # las 2 trazas de leyenda van antes de las N×4 de datos

    for i, (_, row) in enumerate(cands.iterrows()):
        vis     = (i == 0)
        ticker  = str(row["ticker"])
        company = str(row.get("company") or "Sin datos CNV")
        bucket  = str(row.get("riesgo_bucket", "sin dato"))
        _risk_rgba = {
            "bajo (≤2x)": "rgba(39,174,96,0.15)", "moderado (2-4x)": "rgba(243,156,18,0.18)",
            "alto (4-6x)": "rgba(230,126,34,0.18)", "muy alto (>6x)": "rgba(231,76,60,0.18)",
            "EBITDA negativo": "rgba(142,68,173,0.18)", "sin dato": "rgba(149,165,166,0.18)",
        }
        rcolor = _risk_rgba.get(bucket, "rgba(149,165,166,0.18)")

        # ── tabla empresa ────────────────────────────────────────────────────
        anual_flag = "anual ✓" if row.get("deuda_ebitda_es_anual") else "trimestral ⚠"
        fig.add_trace(go.Table(
            header=dict(
                values=["<b>Indicador</b>", "<b>Valor</b>"],
                fill_color=_HDR, font=dict(color="white", size=11),
                align="left", height=26,
            ),
            cells=dict(
                values=[
                    ["Empresa", "Riesgo crediticio", "Deuda Fin / EBITDA",
                     "EBITDA / Intereses", "Liquidez", "Último balance"],
                    [
                        company[:38],
                        bucket,
                        f"{_nd(row.get('deuda_fin_sobre_ebitda'), 'x')} ({anual_flag})",
                        _nd(abs(row["ebitda_sobre_intereses"]) if pd.notna(row.get("ebitda_sobre_intereses")) else float("nan"), "x"),
                        _nd(row.get("liquidez")),
                        str(row.get("fecha_balance_ultimo", "N/D"))[:10],
                    ],
                ],
                fill_color=[[_CELL] * 6, [_CELL, rcolor] + [_CELL] * 4],
                align="left", font=dict(size=10), height=22,
            ),
            visible=vis, columnwidth=[1.3, 1.2],
        ), row=1, col=1)

        # ── tabla bono ────────────────────────────────────────────────────────
        desc_s = str(row.get("descripcion", ""))
        if len(desc_s) > 44:
            desc_s = desc_s[:42] + "…"
        fig.add_trace(go.Table(
            header=dict(
                values=["<b>Campo</b>", "<b>Valor</b>"],
                fill_color=_HDR, font=dict(color="white", size=11),
                align="left", height=26,
            ),
            cells=dict(
                values=[
                    ["Ticker", "Descripción", "Vencimiento",
                     "Precio", "TIR", "Cotiza en", "Volumen op."],
                    [
                        f"<b>{ticker}</b>",
                        desc_s,
                        str(row["fechaVencimiento"])[:10],
                        _nd(row.get("price")),
                        f"<b>{row['tir_pct']:.2f} %</b>",
                        str(row.get("moneda_cotizacion", "")),
                        f"{row['volumen']:,.0f}" if pd.notna(row.get("volumen")) else "N/D",
                    ],
                ],
                fill_color=[[_CELL] * 7],
                align="left", font=dict(size=10), height=22,
            ),
            visible=vis, columnwidth=[1.0, 1.6],
        ), row=1, col=2)

        # ── barras de flujo ───────────────────────────────────────────────────
        fl = _parse_flujos(row["flujos"])
        has_bd = (fl["amortizacion"] > 0).any() or (fl["interes"] > 0).any()
        y_int  = fl["interes"]       if has_bd else pd.Series(dtype=float)
        y_amrt = fl["amortizacion"]  if has_bd else fl["total"]

        for y_vals, color, group, htmpl in [
            (y_int,  _C_INT, "interes", "<b>%{x|%d %b %Y}</b><br>Interés: <b>%{y:.4f}</b><extra></extra>"),
            (y_amrt, _C_AMT, "amort",   "<b>%{x|%d %b %Y}</b><br>Amortización: <b>%{y:.4f}</b><extra></extra>"),
        ]:
            fig.add_trace(go.Bar(
                x=fl["fecha"], y=y_vals,
                marker_color=color,
                legendgroup=group, showlegend=False,
                visible=vis,
                hovertemplate=htmpl,
            ), row=2, col=1)

    # ── dropdown: LEGEND_OFFSET + N×4 trazas ─────────────────────────────────
    buttons = []
    for i, (_, row) in enumerate(cands.iterrows()):
        vis_arr = [True, True] + [False] * (n * 4)   # leyenda siempre True
        vis_arr[LEGEND_OFFSET + i * 4]     = True   # tabla empresa
        vis_arr[LEGEND_OFFSET + i * 4 + 1] = True   # tabla bono
        vis_arr[LEGEND_OFFSET + i * 4 + 2] = True   # bar interes
        vis_arr[LEGEND_OFFSET + i * 4 + 3] = True   # bar amort
        vto = str(row["fechaVencimiento"])[:10]
        buttons.append(dict(
            label=f"{row['ticker']} · {row['tir_pct']:.1f}% · {vto}",
            method="update",
            args=[
                {"visible": vis_arr},
                {"title": f"Dashboard — {row['ticker']}: {str(row.get('descripcion', ''))[:60]}"},
            ],
        ))

    first = cands.iloc[0]
    fig.update_layout(
        title=f"Dashboard — {first['ticker']}: {str(first.get('descripcion', ''))[:60]}",
        height=820,
        template="plotly_white",
        barmode="stack",
        legend=dict(orientation="h", x=0.38, y=0.265, bgcolor="rgba(0,0,0,0)"),
        margin=dict(t=130, b=50, l=60, r=30),
        updatemenus=[dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.0, y=1.13,
            xanchor="left", yanchor="top",
            bgcolor="#f5f8fa",
            bordercolor=_HDR,
            borderwidth=1,
            font=dict(size=11),
            pad=dict(t=4, b=4, l=6, r=6),
        )],
        annotations=[dict(
            text="<b>Seleccioná una ON:</b>",
            x=-0.01, y=1.155,
            xref="paper", yref="paper",
            showarrow=False, font=dict(size=11), align="left",
        )],
    )
    fig.update_xaxes(title_text="Fecha de pago", tickformat="%b %Y", dtick="M6", row=2, col=1)
    fig.update_yaxes(title_text="USD por c/100 VN", row=2, col=1)
    return fig


fig_on = build_on_dashboard(candidatas)
fig_on.show()


## 8. Reporte HTML por ON

Genera un HTML estático con análisis completo de cada candidata: empresa emisora, situación financiera, specs del bono, monto mínimo de inversión y flujo de pagos interactivo.

In [ ]:
from on_report import build_html_report
from pathlib import Path

out_path = Path("../output/on_candidatas_report.html")
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(build_html_report(candidatas), encoding="utf-8")

print(f"Reporte guardado: {out_path.resolve()}")
print(f"Tamaño: {out_path.stat().st_size / 1024:.1f} KB  |  ONs incluidas: {len(candidatas)}")